In [1]:
import argparse
import datetime
import math
import os
from time import time
from typing import Any, List, Tuple

import mlflow
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from sparsemax import Sparsemax

from artifacts import (
    build_plot_df_wrapper,
    save_csv_artifact,
    save_plot_strategy,
)
from MOBO_v2 import MOBO
from data import (
    apply_final_treatment,
    join_metadata,
    load_metadata_artefacts,
    load_odds,
)
from dependencies.config import load_config
from dependencies.utils import get_bet_return, save_df_as_parquet, softmax, sparsemax
from filter import filter_by_linear_combination
from GameProbs import GameProbs

config = load_config("config/config.yml")

sparsemax = Sparsemax(dim=-1)

from loguru import logger


/Users/marcosbarbosa/anaconda3/envs/soccer_betting_strategy_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def setup(args):
    metadata, gameid_to_outcome = load_metadata_artefacts(config.metadata_path)
    odds = load_odds(config.odds_path, args.bookmakers)
    print(odds.shape)
    odds = join_metadata(odds, metadata)

    odds = odds.sort_values(["Datetime", "GameId"], ascending=True)

    #odds = odds[(odds.Datetime.apply(str)>"2019-08-01")&(odds.Datetime.apply(str)<"2019-09-01")]
    #odds = odds[(odds.Datetime.apply(str)>"2019-01-01")&(odds.Datetime.apply(str)<"2020-01-01")]
    odds = odds[(odds.Datetime.apply(str)>="2024-06-02")]
    
    # odds = odds[
    #     (odds.Datetime.apply(str) > "2019-06-01")
    #     & (odds.Datetime.apply(str) < "2019-07-01")
    # ]

    # odds = odds[odds]

    return odds, gameid_to_outcome


def process_group(
    group: Tuple[str, pd.DataFrame], gameid_to_outcome, args
) -> List[List[Any]]:
    is_valid_solution = True

    date, group_data = group

    games_ids = group_data["GameId"].unique()

    # Initialize dict to store dataframes of favorable bet opportunities
    odds_dict = {}
    # Initialize dict to store 7x7 matrices/dataframes of real probabilities
    df_probs_dict = {}

    if len(games_ids) > args.min_games:
        for game_id in games_ids:
            df = GameProbs(game_id).build_dataframe()

            odds_sample = group_data[(group_data.GameId == game_id)]
            odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
            if not args.do_baseline:
                odds_sample = filter_by_linear_combination(odds_sample, n=args.bets_per_game, weight=args.weight)
            else:
                odds_sample = odds_sample.sample(1)
            odds_dict[game_id] = odds_sample
            df_probs_dict[game_id] = df

        odds_dt = pd.concat(odds_dict.values())

        if len(odds_dt) <= config.max_vector_length and len(odds_dt) > 1:
        #if len(odds_dt) <= config.max_vector_length :
            iteration_date = odds_dt.Datetime.apply(str).unique()[0]
            print(f"Date: {iteration_date}")

            odds_favorable = torch.tensor(np.array(odds_dt["Odd"]))
            real_prob_favorable = torch.tensor(np.array(odds_dt["real_prob"]))
            event_favorable = list(odds_dt["BetMap"].values)
            games_ids = np.array(odds_dt["GameId"])
            time_limit_flag = None

            if not args.do_baseline:
                # try:
                print("Execution of minimization task...")


                optimizer_instance = MOBO(
                    n_iterations=args.n_iterations,
                    public_odd=odds_favorable,
                    real_probabilities=real_prob_favorable,
                    event=event_favorable,
                    games_ids=games_ids,
                    df_probs_dict=df_probs_dict,
                )

                _, _, solution = optimizer_instance.run_optimization()

                solution = solution[0]
                print("Finalization of minimization task...")

                # except ValueError:
                # continue

                if any(math.isnan(x) for x in solution):
                    is_valid_solution = False
                odds_dt["solution"] = sparsemax(torch.tensor(np.array([solution]))).tolist()[0]
                #odds_dt["solution"] = softmax(solution)

            else:
                odds_dt["solution"] = 1

            save_df_as_parquet(odds_dt, str(date))

            track_record = []

            for game_id, game_data in odds_dt.groupby("GameId", sort=False):
                scenario = gameid_to_outcome[game_id]
                financial_return = get_bet_return(
                    df=game_data, allocation_array=game_data.solution, scenario=scenario
                )

                print(
                    f"game_id: {game_id}; financial_return: {np.round(financial_return, 3)}"
                )

                track_record.append(
                    [
                        str(game_id),
                        financial_return,
                        len(game_data),
                        odds_dt.n_favorable_bets.values[0],
                        time_limit_flag,
                        is_valid_solution,
                        iteration_date,
                    ]
                )

            return track_record

In [3]:
#args = parser.parse_args()
parser = argparse.ArgumentParser()
parser.add_argument(
    '--bookmakers',
    nargs='+',
    default=None,
    help='A list of strings',
)
parser.add_argument(
    "--aggregator", type=str, help="aggregate by GameId or by Datetime"
)
parser.add_argument(
    "--min_games",
    type=int,
    default=0,
    help="threshold of minimum number of games to enter the optimization task",
)
parser.add_argument(
    "--bets_per_game", type=int, default=5, help="number of bets per game"
)
parser.add_argument(
    "--weight",
    type=float,
    default=0.5,
    help="weight of the linear combination filter",
)
parser.add_argument(
    "--n_iterations",
    type=int,
    default=10,
    help="number of iterations to run the optimization task",
)
parser.add_argument(
    "--do_baseline",
    action="store_true",
    help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action",
)
parser.add_argument(
    "--n_jobs",
    type=int,
    default=1,
    help="number of jobs to run in parallel",
)
parser.add_argument(
    "--save_experiment",
    action="store_true",
    help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action",
)
args = parser.parse_args([
"--aggregator", "Datetime",
"--min_games", "1",
"--n_iterations", "100",
"--weight", "0.2",
#"--do_baseline", "False"
#"--save_experiment"
])
print(args)

odds, gameid_to_outcome = setup(args)

grouped = odds.groupby(args.aggregator)

Namespace(bookmakers=None, aggregator='Datetime', min_games=1, bets_per_game=5, weight=0.2, n_iterations=100, do_baseline=False, n_jobs=1, save_experiment=False)
(2113785, 7)


In [4]:
args.weight

0.2

In [5]:
for a, b in grouped:
    print(a)
    print(b)

2024-06-02
          GameId      Sportsbook Market Scenario   Bet     Odd  public_prob  \
2112391  8581153       Stake.com    h2h     None  home    2.16     0.462963   
2112392  8581153       Stake.com    h2h     None  draw    3.20     0.312500   
2112393  8581153       Stake.com    h2h     None  away    3.50     0.285714   
2112394  8581153   BC.Game Sport    h2h     None  home    2.15     0.465116   
2112395  8581153   BC.Game Sport    h2h     None  draw    3.20     0.312500   
...          ...             ...    ...      ...   ...     ...          ...   
2113780  8581158           20Bet  exact    6 : 0   odd  101.00     0.009901   
2113781  8581158  LeoVegas Sport  exact    6 : 1   odd  301.00     0.003322   
2113782  8581158         Betobet  exact    6 : 2   odd  501.00     0.001996   
2113783  8581158         Mostbet  exact    7 : 0   odd  450.00     0.002222   
2113784  8581158         Mostbet  exact    7 : 1   odd  500.00     0.002000   

                Home      Away    Dateti

In [6]:
# Function to get the n-th group and its DataFrame
def get_nth_group(grouped, n):
    for i, (group_key, group_df) in enumerate(grouped):
        if i == n:
            return group_key, group_df
    raise IndexError("Group index out of range")
n = 0
group = get_nth_group(grouped, n)

In [7]:
date, group_data = group
date

datetime.date(2024, 6, 2)

In [8]:

games_ids = group_data["GameId"].unique()

# Initialize dict to store dataframes of favorable bet opportunities
odds_dict = {}
# Initialize dict to store 7x7 matrices/dataframes of real probabilities
df_probs_dict = {}

if len(games_ids) > args.min_games:
    for game_id in games_ids:
        df = GameProbs(game_id).build_dataframe()

        odds_sample = group_data[(group_data.GameId == game_id)]
        odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
        if not args.do_baseline:
            odds_sample = filter_by_linear_combination(odds_sample, n=args.bets_per_game, weight=args.weight)
        else:
            odds_sample = odds_sample.sample(1)
        odds_dict[game_id] = odds_sample
        df_probs_dict[game_id] = df

    odds_dt = pd.concat(odds_dict.values())

    if len(odds_dt) <= config.max_vector_length and len(odds_dt) > 1:
    #if len(odds_dt) <= config.max_vector_length :
        iteration_date = odds_dt.Datetime.apply(str).unique()[0]
        print(f"Date: {iteration_date}")

        odds_favorable = torch.tensor(np.array(odds_dt["Odd"]))
        real_prob_favorable = torch.tensor(np.array(odds_dt["real_prob"]))
        event_favorable = list(odds_dt["BetMap"].values)
        games_ids = np.array(odds_dt["GameId"])
        time_limit_flag = None

        if not args.do_baseline:
            # try:
            print("Execution of minimization task...")


            optimizer_instance = MOBO(
                n_iterations=args.n_iterations,
                public_odd=odds_favorable,
                real_probabilities=real_prob_favorable,
                event=event_favorable,
                games_ids=games_ids,
                df_probs_dict=df_probs_dict,
            )

            _, _, set_solution = optimizer_instance.run_optimization()


            solution = set_solution[0]
            print("Finalization of minimization task...")

            # except ValueError:
            # continue

            if any(math.isnan(x) for x in solution):
                is_valid_solution = False
            odds_dt["solution"] = sparsemax(torch.tensor(np.array([solution]))).tolist()[0]
            #odds_dt["solution"] = softmax(solution)

        else:
            odds_dt["solution"] = 1

        track_record = []
            
        financial_return_aggregated = 0

        for game_id, game_data in odds_dt.groupby("GameId", sort=False):
            scenario = gameid_to_outcome[game_id]
            financial_return = get_bet_return(
                df=game_data, allocation_array=game_data.solution, scenario=scenario
            )
            financial_return_aggregated += financial_return
            logger.info(
                f"scenario {scenario}; game_id: {game_id}; financial_return: {np.round(financial_return, 3)}"
            )

Date: 2024-06-02
Execution of minimization task...


/Users/marcosbarbosa/anaconda3/envs/soccer_betting_strategy_env/lib/python3.9/site-packages/botorch/models/transforms/outcome.py:304: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1760.)
  stdvs = Y.std(dim=-2, keepdim=True)
/Users/marcosbarbosa/anaconda3/envs/soccer_betting_strategy_env/lib/python3.9/site-packages/botorch/models/utils/assorted.py:194: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1760.)
  Ymean, Ystd = torch.mean(Y, dim=-2), torch.std(Y, dim=-2)
/Users/marcosbarbosa/anaconda3/envs/soccer_betting_strategy_env/lib/python3.9/site-packages/botorch/optim/initializers.py:

Finalization of minimization task...


In [17]:
odds_dt

,GameId,Sportsbook,Market,Scenario,Bet,Odd,public_prob,Home,Away,Datetime,BetMap,real_prob,bet_flag,n_favorable_bets,odd_dist,score,expected_return,solution
4,8581153,Stake.com,over/under,2.5,over,2.210,0.452489,Corinthians,Botafogo,2024-06-02,"[0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, ...",0.8766,True,12,0.5,0.80128,1.937286,0.000000
6,8581153,Fezbet,over/under,3.5,over,4.200,0.238095,Corinthians,Botafogo,2024-06-02,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, ...",0.8231,True,12,0.7,0.79848,3.457020,0.037806
2,8581153,Stake.com,over/under,1.5,over,1.390,0.719424,Corinthians,Botafogo,2024-06-02,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...",0.9476,True,12,0.2,0.79808,1.317164,0.082184
0,8581153,Fezbet,over/under,0.5,over,1.070,0.934579,Corinthians,Botafogo,2024-06-02,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.9588,True,12,0.0,0.76704,1.025916,0.070038
8,8581153,Fezbet,over/under,4.5,over,8.500,0.117647,Corinthians,Botafogo,2024-06-02,"[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, ...",0.7228,True,12,0.8,0.73824,6.143800,0.015111
2,8581154,Stake.com,over/under,1.5,over,1.430,0.699301,Criciúma,Palmeiras,2024-06-02,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...",0.9017,True,12,0.2,0.76136,1.289431,0.000000
0,8581154,Fezbet,over/under,0.5,over,1.080,0.925926,Criciúma,Palmeiras,2024-06-02,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.9480,True,12,0.0,0.75840,1.023840,0.000480
4,8581154,Stake.com,over/under,2.5,over,2.310,0.432900,Criciúma,Palmeiras,2024-06-02,"[0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, ...",0.7782,True,12,0.4,0.70256,1.797642,0.007625
19,8581154,bet365,spread,-1/+1,away,1.182,0.846024,Criciúma,Palmeiras,2024-06-02,"[1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, ...",0.8588,True,12,0.0,0.68704,1.015102,0.070128
6,8581154,Stake.com,over/under,3.5,over,3.450,0.289855,Criciúma,Palmeiras,2024-06-02,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, ...",0.6689,True,12,0.6,0.65512,2.307705,0.000000


In [74]:
odds_dt.solution.sum()

1.0000000000000002

In [75]:
solution

array([0.02390614, 0.02064826, 0.0441017 , 0.03712869, 0.08786532,
       0.05564704, 0.02816592, 0.04842948, 0.09071292, 0.06418699,
       0.06667573, 0.00188484, 0.09123806, 0.09241454, 0.01231877,
       0.03806763, 0.06941284, 0.05102071])

In [ ]:

def run_strategy(args):
    start_time = time()

    odds, gameid_to_outcome = setup(args)

    grouped = odds.groupby(args.aggregator)
    print(f"The number of jobs is: {args.n_jobs}")
    # Parallelize the group processing
    results = Parallel(n_jobs=args.n_jobs)(
        delayed(process_group)(group, gameid_to_outcome, args) for group in grouped
    )

    data = [x for x in results if x is not None]
    df_flat = pd.DataFrame([item for sublist in data for item in sublist])

    # # Start an MLflow experiment
    # with mlflow.start_run():
    #     # Log parameters (e.g., settings of the optimizer)
    #     mlflow.log_param("aggregator", args.aggregator)
    #     mlflow.log_param("min_games", args.min_games)
    #     mlflow.log_param("bookmakers", args.bookmakers)
    #     mlflow.log_param("bets_per_game", args.bets_per_game)
    #     mlflow.log_param("weight", args.weight)
    #     mlflow.log_param("do_baseline", args.do_baseline)
    #     mlflow.log_param("n_iterations", args.n_iterations)


    #     if args.save_experiment:
    #         # Create artefacts folder
    #         timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    #         artefacts_folder = f"artefacts/{timestamp}"
    #         os.makedirs(artefacts_folder)
    #         save_csv_artifact(artefacts_folder, "result", df_flat)
    #         df_plot = build_plot_df_wrapper(artefacts_folder,  args.aggregator, args.do_baseline)
    #         save_csv_artifact(artefacts_folder, "result_plot", df_plot)
    #         save_plot_strategy(artefacts_folder, df_plot)
        
    #     mlflow.log_param("timestamp", timestamp)
    #     mlflow.log_artifact(f"{artefacts_folder}/result_plot.csv")
    #     mlflow.log_artifact(f"{artefacts_folder}/plot.PNG")
    #     mlflow.log_metric("wealth", df_plot["stake"].values[-1])

    #     # End the MLflow run
    #     mlflow.end_run()

    # elapsed_time = time() - start_time
    # print("Final Elapsed: %.3f sec" % elapsed_time)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        '--bookmakers',
        nargs='+',
        default=None,
        help='A list of strings',
    )
    parser.add_argument(
        "--aggregator", type=str, help="aggregate by GameId or by Datetime"
    )
    parser.add_argument(
        "--min_games",
        type=int,
        default=0,
        help="threshold of minimum number of games to enter the optimization task",
    )
    parser.add_argument(
        "--bets_per_game", type=int, default=5, help="number of bets per game"
    )
    parser.add_argument(
        "--weight",
        type=float,
        default=0.5,
        help="weight of the linear combination filter",
    )
    parser.add_argument(
        "--n_iterations",
        type=int,
        default=10,
        help="number of iterations to run the optimization task",
    )
    parser.add_argument(
        "--do_baseline",
        action="store_true",
        help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action",
    )
    parser.add_argument(
        "--n_jobs",
        type=int,
        default=1,
        help="number of jobs to run in parallel",
    )
    parser.add_argument(
        "--save_experiment",
        action="store_true",
        help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action",
    )
    args = parser.parse_args()
    print(args)
    run_strategy(args)
